## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
DDR_ZIP = "/content/drive/MyDrive/DDR.zip"     # change to the real path of your IDRiD zip in Drive

if not os.path.exists("/content/ddr/DDR"):
    !unzip -q "{DDR_ZIP}" -d /content/ddr
print(os.listdir("/content/ddr"))

['DR_grading.csv', 'DR_grading']


In [ ]:
!pip install timm pennylane pennylane-lightning -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 17.6 MB/s eta 0:00:00


## 2. Imports and configuration

In [ ]:
import os, random, hashlib
import numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import timm
import pennylane as qml
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             cohen_kappa_score, roc_auc_score, classification_report, confusion_matrix)

In [ ]:
BASE_PATH = "/content/drive/MyDrive/your folder"
os.makedirs(BASE_PATH + "/checkpoints", exist_ok=True)

SEED = 42          # change to 43, 44 for extra seeds

config = {
    "epochs": 50,
    "batch_size": 32,
    "num_classes": 5,
    "lr_backbone": 1e-4,
    "lr_other": 1e-4,
    "lr_alpha": 1e-2,        # raised so alpha can really move (APTOS gives only ~88 steps per epoch)
    "weight_decay": 0.01,

    # ---- experiment switches: the defaults below are the BASELINE model ----
    "quantum_mode": "circuit",   # "circuit" | "skip" (no circuit, same 8 numbers) | "mlp" (matched classical layer) | "frozen" (random untrained circuit)
    "fixed_alpha": None,         # None = learnable. Or a number (0.0, 0.27, 0.5, 1.0) to fix the mixing weight
    "use_pos_emb": False,        # True = add learned position/scale embeddings to the tokens
    "n_trans_layers": 4,         # Transformer depth (paper text says 2 - decide using validation results)

    "tag": "v3",                   # e.g. "smoke" for the quick test run
}

RUN = (f"{config['tag']}_" if config["tag"] else "") + \
      f"{config['quantum_mode']}_a{config['fixed_alpha']}_pe{int(config['use_pos_emb'])}_L{config['n_trans_layers']}_s{SEED}"
print("RUN name:", RUN)

RUN name: v3_circuit_aNone_pe0_L4_s42


## 5. Quantum circuit (8 qubits, 3 layers, data re-uploading, ring CNOT)

In [ ]:
n_qubits = 8
n_layers = 3

dev = qml.device("default.qubit", wires=n_qubits)

def quantum_circuit(inputs, weights):
    for i in range(n_qubits):                       # initial U3 encoding
        qml.U3(inputs[..., i], inputs[..., i], inputs[..., i], wires=i)
    for l in range(n_layers):
        for i in range(n_qubits):                   # data re-uploading
            qml.RY(inputs[..., i], wires=i)
        for i in range(n_qubits):                   # trainable single-qubit rotations (72 parameters in total)
            qml.U3(weights[l][i][0], weights[l][i][1], weights[l][i][2], wires=i)
        for i in range(n_qubits):                   # ring entanglement
            qml.CNOT(wires=[i, (i + 1) % n_qubits])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (n_layers, n_qubits, 3)}
qnode = qml.QNode(quantum_circuit, dev, interface="torch", diff_method="backprop")

In [ ]:
class QuantumBranch(nn.Module):
    """256 -> attention gate -> reducer -> 8 angles -> (circuit | control) -> 8 features"""
    def __init__(self, feature_dim=256, mode="circuit"):
        super().__init__()
        self.mode = mode

        self.attn = nn.Sequential(
            nn.Linear(feature_dim, 64), nn.ReLU(), nn.Linear(64, feature_dim)
        )
        self.reducer = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 128),         nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 64),          nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, n_qubits)
        )

        if mode in ("circuit", "frozen"):
            self.quantum = qml.qnn.TorchLayer(qnode, weight_shapes)   # a fresh circuit for every model
            if mode == "frozen":                                      # random circuit that is never trained
                for p in self.quantum.parameters():
                    p.requires_grad_(False)
        elif mode == "mlp":                                           # matched classical layer: 8*8 + 8 = 72 parameters
            self.quantum = nn.Sequential(nn.Linear(n_qubits, n_qubits), nn.Tanh())
        elif mode == "skip":                                          # no circuit at all
            self.quantum = None
        else:
            raise ValueError(f"unknown quantum_mode: {mode}")

    def forward(self, x):
        a = torch.softmax(self.attn(x), dim=-1)
        x = x * a
        x = self.reducer(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = torch.tanh(x) * np.pi                       # 8 angles in (-pi, pi)

        if self.mode in ("circuit", "frozen"):
            q_out = self.quantum(x.float())
        elif self.mode == "mlp":
            q_out = self.quantum(x.float() / np.pi)
        else:                                            # "skip": same 8 numbers, scaled to (-1, 1) like the circuit output
            q_out = x.float() / np.pi
        return q_out, x

In [ ]:
class HybridMSLATNet(nn.Module):
    def __init__(self, num_classes=5, embed_dim=256, n_trans_layers=4, use_pos_emb=False,
                 quantum_mode="circuit", fixed_alpha=None):
        super().__init__()

        self.backbone = timm.create_model("efficientnet_b2", pretrained=True, features_only=True)
        feat_dims = [f["num_chs"] for f in self.backbone.feature_info[-3:]]

        self.proj = nn.ModuleList([nn.Linear(feat_dims[i], embed_dim) for i in range(3)])

        # optional learned position / scale embeddings (one vector per token of the concatenated sequence)
        self.pos_emb = None
        if use_pos_emb:
            was_training = self.backbone.training
            self.backbone.eval()
            with torch.no_grad():
                fs = self.backbone(torch.zeros(1, 3, 224, 224))[-3:]
            n_tokens = sum(f.shape[2] * f.shape[3] for f in fs)
            self.backbone.train(was_training)
            self.pos_emb = nn.Parameter(torch.zeros(1, n_tokens, embed_dim))
            nn.init.trunc_normal_(self.pos_emb, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=8, dim_feedforward=embed_dim * 4,
            dropout=0.5, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_trans_layers, enable_nested_tensor=False)

        self.pool = nn.Linear(embed_dim, 1)

        self.classical_head = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

        self.quantum_branch = QuantumBranch(embed_dim, mode=quantum_mode)

        self.fusion_head = nn.Sequential(
            nn.Linear(embed_dim + n_qubits, 256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        self.quantum_head = nn.Sequential(
            nn.Linear(n_qubits, 32), nn.GELU(), nn.Linear(32, num_classes)
        )

        # mixing weight between the classical and the fused prediction
        self.learn_alpha = fixed_alpha is None
        if self.learn_alpha:
            self.alpha = nn.Parameter(torch.tensor(-1.0))               # sigmoid(-1) = 0.269 at the start
        else:
            self.register_buffer("alpha", torch.tensor(float(fixed_alpha)))

    def get_alpha(self):
        return torch.sigmoid(self.alpha) if self.learn_alpha else self.alpha

    def forward(self, x):
        feats = self.backbone(x)[-3:]

        tokens = []
        for i, f in enumerate(feats):
            t = f.flatten(2).transpose(1, 2)          # (B, H*W, C): every spatial position is one token
            tokens.append(self.proj[i](t))
        tokens = torch.cat(tokens, dim=1)
        if self.pos_emb is not None:
            tokens = tokens + self.pos_emb
        tokens = self.transformer(tokens)

        attn = torch.softmax(self.pool(tokens), dim=1)
        pooled = (tokens * attn).sum(dim=1)            # (B, 256)

        classical_logits = self.classical_head(pooled)

        quantum_features, q_input = self.quantum_branch(pooled)
        fusion_logits  = self.fusion_head(torch.cat([pooled, quantum_features], dim=1))
        quantum_logits = self.quantum_head(quantum_features)

        alpha = self.get_alpha()
        out = (1 - alpha) * classical_logits + alpha * fusion_logits

        return out, classical_logits, quantum_logits, pooled, attn, q_input

In [ ]:
import os
print(os.listdir("/content/ddr"))
!df -h /content | tail -1

['DR_grading.csv', 'DR_grading']
overlay         113G   49G   64G  44% /


In [ ]:
for pre in ["v3_circuit_aNone_pe0_L4", "skip50_skip_aNone_pe0_L4"]:
    for s in (42, 43, 44):
        f = f"{BASE_PATH}/checkpoints/best_{pre}_s{s}.pth"
        print(os.path.basename(f), "OK" if os.path.exists(f) else "MISSING")

best_v3_circuit_aNone_pe0_L4_s42.pth OK
best_v3_circuit_aNone_pe0_L4_s43.pth OK
best_v3_circuit_aNone_pe0_L4_s44.pth OK
best_skip50_skip_aNone_pe0_L4_s42.pth OK
best_skip50_skip_aNone_pe0_L4_s43.pth OK
best_skip50_skip_aNone_pe0_L4_s44.pth OK


In [ ]:
import os, re, numpy as np, pandas as pd
DDR_DIR = "/content/ddr"
IMG_EXT = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")

# ---- index every image ----
by_name, by_stem = {}, {}
for root, _, files in os.walk(DDR_DIR):
    for f in files:
        if f.lower().endswith(IMG_EXT):
            p = os.path.join(root, f)
            by_name.setdefault(f, []).append(p); by_stem.setdefault(os.path.splitext(f)[0], []).append(p)
print("images found:", sum(len(v) for v in by_name.values()), "| distinct file names:", len(by_name))

def pick(paths):
    if len(paths) == 1: return paths[0]
    clean = [p for p in paths if not re.search(r"mask|segment|lesion|label", p, re.I)] or paths
    grad = [p for p in clean if re.search(r"grad", p, re.I)]
    return (grad or clean)[0]

# ---- read the grading csv ----
csvs = sorted(os.path.join(r, f) for r, _, fs in os.walk(DDR_DIR) for f in fs if f.lower().endswith(".csv"))
print("csv files:", csvs)
lab = pd.read_csv(csvs[0]); print("using", csvs[0], lab.shape); print(lab.head(3).to_string())
cols = list(lab.columns)
name_col = cols[0]
lab_col = next((c for c in cols[1:] if re.search(r"level|grade|label|diagnos|^dr", str(c), re.I)), cols[-1])
split_col = next((c for c in cols if re.search(r"split|subset|partition|phase|^set$|fold", str(c), re.I)), None)
if split_col is None:
    for c in cols[1:]:
        if lab[c].astype(str).str.lower().isin(["train", "training", "val", "valid", "validation", "test"]).mean() > 0.9: split_col = c
print("name column:", name_col, "| label column:", lab_col, "| split column:", split_col)

def norm_split(s):
    s = str(s).lower()
    if "test" in s: return "test"
    if "val" in s: return "val"
    if "train" in s: return "train"
    return None

def split_from_path(p):
    for part in reversed(os.path.dirname(p).split(os.sep)):
        sp = norm_split(part)
        if sp: return sp
    return None

recs = []
for _, r in lab.iterrows():
    n = str(r[name_col]).strip()
    paths = by_name.get(n) or by_stem.get(os.path.splitext(n)[0])
    path = pick(paths) if paths else None
    sp = norm_split(r[split_col]) if split_col else (split_from_path(path) if path else None)
    try: g = int(r[lab_col])
    except Exception: g = None
    recs.append(dict(split=sp, name=n, grade=g, path=path))
ddr_all = pd.DataFrame(recs)
print("rows:", len(ddr_all), "| no matching image:", int(ddr_all.path.isna().sum()), "| split unknown:", int(ddr_all.split.isna().sum()))
print("\nraw grade counts per split (0-4 = DR grade, 5 = ungradable):")
print(pd.crosstab(ddr_all.split.fillna("unknown"), ddr_all.grade).to_string())

ddr = ddr_all[ddr_all.grade.between(0, 4) & ddr_all.path.notna() & ddr_all.split.notna()].rename(columns={"grade": "diagnosis"}).reset_index(drop=True)
ddr["diagnosis"] = ddr["diagnosis"].astype(int)
if ddr.empty and ddr_all.split.isna().all():      # no split information anywhere in this copy of DDR
    ddr = ddr_all[ddr_all.grade.between(0, 4) & ddr_all.path.notna()].assign(split="all").rename(columns={"grade": "diagnosis"}).reset_index(drop=True)
    ddr["diagnosis"] = ddr["diagnosis"].astype(int)
    print("\nNO SPLIT INFORMATION FOUND -> all gradable images are labelled split='all'. In cell D2 set EVAL_SPLIT = 'all'.")
print("\nafter dropping ungradable / missing / unknown split:", ddr.groupby("split").size().to_dict(),
      "(about 6320 / 2503 / 3759 for train / val / test in the original release)")

images found: 12524 | distinct file names: 12524
csv files: ['/content/ddr/DR_grading.csv']
using /content/ddr/DR_grading.csv (12522, 2)
                 id_code  diagnosis
0  20170413102628830.jpg          0
1  20170413111955404.jpg          0
2  20170413112015395.jpg          0
name column: id_code | label column: diagnosis | split column: None
rows: 12522 | no matching image: 0 | split unknown: 12522

raw grade counts per split (0-4 = DR grade, 5 = ungradable):
grade       0    1     2    3    4
split                             
unknown  6266  630  4477  236  913

NO SPLIT INFORMATION FOUND -> all gradable images are labelled split='all'. In cell D2 set EVAL_SPLIT = 'all'.

after dropping ungradable / missing / unknown split: {'all': 12522} (about 6320 / 2503 / 3759 for train / val / test in the original release)


In [ ]:
import cv2, numpy as np, pandas as pd, torch, time
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score, confusion_matrix

EVAL_SPLIT = "all"
df = ddr[ddr.split == EVAL_SPLIT].reset_index(drop=True)
# df = df.sample(4000, random_state=0).reset_index(drop=True)   # remove the leading '#' ONLY if the time estimate is too long
Y = df.diagnosis.values
print(EVAL_SPLIT, len(df), "images | grade counts:", np.bincount(Y, minlength=5))

val_tf = A.Compose([A.Resize(224, 224), A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), ToTensorV2()])
class DDRDS(Dataset):
    def __init__(self, d): self.p, self.y = d.path.tolist(), d.diagnosis.tolist()
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        im = cv2.imread(self.p[i])
        if im is None: raise RuntimeError("cannot read " + self.p[i])
        im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_AREA)       # same 256x256 step as APTOS and IDRiD
        return val_tf(image=cv2.cvtColor(im, cv2.COLOR_BGR2RGB))["image"], self.y[i]
loader = DataLoader(DDRDS(df), batch_size=32, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); use_amp = device.type == "cuda"
print("device:", device)
runs = [("circuit", f"v3_circuit_aNone_pe0_L4_s{s}") for s in (42, 43, 44)] + \
       [("skip", f"skip50_skip_aNone_pe0_L4_s{s}") for s in (42, 43, 44)]
models = {}
for mode, RUNN in runs:
    ck = torch.load(f"{BASE_PATH}/checkpoints/best_{RUNN}.pth", map_location=device, weights_only=False)
    m = HybridMSLATNet(n_trans_layers=4, use_pos_emb=False, quantum_mode=mode, fixed_alpha=None).to(device)
    m.load_state_dict(ck["model"]); m.eval(); models[RUNN] = m
print("loaded", len(models), "models")

t0 = time.time()
P = {r: [] for _, r in runs}
with torch.no_grad():
    for imgs, _ in tqdm(loader):
        imgs = imgs.to(device, non_blocking=True)
        for _, RUNN in runs:
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = models[RUNN](imgs)[0]
            P[RUNN].append(torch.softmax(out.float(), 1).cpu().numpy())
print("inference time: %.1f min" % ((time.time() - t0) / 60))
P = {r: np.concatenate(v).astype(np.float64) for r, v in P.items()}
P = {r: v / v.sum(1, keepdims=True) for r, v in P.items()}
for r, v in P.items(): np.savez(f"{BASE_PATH}/ext_ddr_{EVAL_SPLIT}_{r}.npz", P=v, Y=Y)

L = [0, 1, 2, 3, 4]
def summarize(y, Pm):
    pr = Pm.argmax(1); cm = confusion_matrix(y, pr, labels=L)
    return dict(acc=accuracy_score(y, pr), f1=f1_score(y, pr, average="macro", labels=L, zero_division=0),
                qwk=cohen_kappa_score(y, pr, weights="quadratic", labels=L),
                auc=roc_auc_score(y, Pm, multi_class="ovr", labels=L),
                ref_auc=roc_auc_score((y >= 2).astype(int), Pm[:, 2:].sum(1)),
                sens=cm[2:, 2:].sum() / cm[2:].sum(), spec=cm[:2, :2].sum() / cm[:2].sum()), cm

rows = []
for mode, RUNN in runs:
    s, _ = summarize(Y, P[RUNN]); rows.append(dict(run=RUNN, family=mode, dataset=f"ddr_{EVAL_SPLIT}", n=len(Y), **s))
    print(RUNN, {k: round(float(v), 4) for k, v in s.items()})
res = pd.DataFrame(rows)
print("\nmean / std by family:")
print(res.groupby("family")[["acc", "f1", "qwk", "auc", "ref_auc", "sens", "spec"]].agg(["mean", "std"]).round(3).to_string())
for fam in ("circuit", "skip"):
    s, cm = summarize(Y, np.mean([P[r] for (md, r) in runs if md == fam], 0))
    print(f"\n{fam} 3-seed ensemble:", {k: round(float(v), 4) for k, v in s.items()}); print(cm)
res.to_csv(f"{BASE_PATH}/results_external_ddr.csv", index=False)
print("\nsaved to", BASE_PATH)

all 12522 images | grade counts: [6266  630 4477  236  913]
device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B / 36.8MB            

model.safetensors: downloading bytes:           |  0.00B            

loaded 6 models


100%|██████████| 392/392 [05:23<00:00,  1.21it/s]


inference time: 5.4 min
v3_circuit_aNone_pe0_L4_s42 {'acc': 0.6087, 'f1': 0.3581, 'qwk': 0.5451, 'auc': 0.7314, 'ref_auc': 0.7772, 'sens': 0.3969, 'spec': 0.9883}
v3_circuit_aNone_pe0_L4_s43 {'acc': 0.6053, 'f1': 0.3719, 'qwk': 0.5548, 'auc': 0.7166, 'ref_auc': 0.7663, 'sens': 0.3987, 'spec': 0.9888}
v3_circuit_aNone_pe0_L4_s44 {'acc': 0.6497, 'f1': 0.3845, 'qwk': 0.6037, 'auc': 0.7617, 'ref_auc': 0.8159, 'sens': 0.5354, 'spec': 0.9693}
skip50_skip_aNone_pe0_L4_s42 {'acc': 0.6083, 'f1': 0.3713, 'qwk': 0.5292, 'auc': 0.7375, 'ref_auc': 0.7839, 'sens': 0.398, 'spec': 0.9871}
skip50_skip_aNone_pe0_L4_s43 {'acc': 0.6283, 'f1': 0.3905, 'qwk': 0.6043, 'auc': 0.7547, 'ref_auc': 0.84, 'sens': 0.4701, 'spec': 0.982}
skip50_skip_aNone_pe0_L4_s44 {'acc': 0.6521, 'f1': 0.3996, 'qwk': 0.5878, 'auc': 0.7271, 'ref_auc': 0.7965, 'sens': 0.5132, 'spec': 0.9748}

mean / std by family:
           acc            f1           qwk           auc        ref_auc          sens          spec       
          mea